# Diet Model: Train + Evaluate with Visualizations

Trains the **actual RandomForest in this notebook** (same code/config as
`python -m ml.train`), then computes precision, recall, F1, confusion matrix,
feature importance, and food-catalog stats — all from the live model and data.

**In Colab:** upload this repo (or `!git clone <your-repo-url>`), then set `REPO` below
to the repo root, e.g. `REPO = "/content/<repo-name>"`. Locally, just run as-is.

In [1]:
from pathlib import Path

def _find_repo():
    here = Path.cwd().resolve()
    for p in [here, *here.parents]:
        if (p / "artifacts" / "diet_model.joblib").exists():
            return p
    return Path("/content/repo")

REPO = _find_repo()
assert (REPO / "artifacts" / "diet_model.joblib").exists(), f"artifacts not found under {REPO}"
print("repo:", REPO)

repo: /home/arnav_kulkarni/Documents/Aiml cie


In [ ]:
# On Arch/PEP-668 systems, NEVER pip-install into system Python.
# Use the repo venv instead (one time):
#   python -m venv .venv && source .venv/bin/activate && pip install -r requirements.txt
# Then select the '.venv' kernel (or run: python -m ipykernel install --user
# --name=repo --display-name='Python (repo)') and skip this cell.
# In Colab this cell is unnecessary (libs preinstalled) — skip it.
import importlib, sys
for _pkg in ["sklearn", "pandas", "matplotlib", "seaborn", "joblib"]:
    try:
        importlib.import_module(_pkg)
        print(f"{_pkg}: ok")
    except ImportError:
        print(f"{_pkg}: MISSING → install into your venv: {sys.executable} -m pip install {_pkg}")


## 1. Load dataset


In [ ]:
df_raw = pd.read_csv(REPO / "datasets" / "diet_recommendations_dataset.csv")
print("rows:", len(df_raw))
print("target classes:", sorted(df_raw["Diet_Recommendation"].unique()))
print("class balance:", df_raw["Diet_Recommendation"].value_counts().to_dict())
print("missing %:", (df_raw.isna().mean() * 100).round(1).to_dict())


## 2. Train the Random Forest here

This trains the **actual model** in this session using the repo's own training
code (`ml.train.build_diet_pipeline`) — same 200-tree configuration, same features,
same 80/20 stratified split (seed 42) as `python -m ml.train`. Nothing is loaded
from disk yet; the saved artifact is only used afterwards as a cross-check.

In [ ]:
import sys, time
sys.path.insert(0, str(REPO))
from ml.data_preparation import diet_features, prepare_diet_frame
from ml.config import DIET_NUMERIC, DIET_CATEGORICAL
from ml.train import build_diet_pipeline
from sklearn.dummy import DummyClassifier
from sklearn.model_selection import train_test_split

# NOTE: raw CSV must go through prepare_diet_frame first (NaN -> 'None' levels),
# exactly as in training — otherwise inputs differ from what the model expects.
df = prepare_diet_frame(REPO / "datasets" / "diet_recommendations_dataset.csv")
X, y = diet_features(df)
print("features:", len(X.columns), "| target:", sorted(y.unique()))
Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.2, random_state=42,
                                        stratify=y)
print("train:", len(Xtr), "| test:", len(yte))

t0 = time.time()
model = build_diet_pipeline(
    [c for c in DIET_NUMERIC if c in X.columns],
    [c for c in DIET_CATEGORICAL if c in X.columns])
model.fit(Xtr, ytr)
print(f"trained RandomForest in {time.time() - t0:.1f}s:", model.named_steps["clf"])

dummy = DummyClassifier(strategy="stratified", random_state=42)
dummy.fit(Xtr, ytr)  # baseline on the SAME split

pred = model.predict(Xte)
labels = sorted(y.unique())
print("test size:", len(yte))


## 3. Evaluate the freshly trained model

Held-out test split only — never training data.

## 4. Precision, recall, F1 per class

In [ ]:
report = classification_report(yte, pred, zero_division=0, output_dict=True)
per_class = pd.DataFrame(
    {k: {"precision": v["precision"], "recall": v["recall"],
            "f1-score": v["f1-score"], "support": v["support"]}
     for k, v in report.items() if k in labels}).T
print(per_class.round(4).to_string())
print()
for avg in ["macro avg", "weighted avg"]:
    v = report[avg]
    print(f"{avg}: precision={v['precision']:.4f} recall={v['recall']:.4f} "
          f"f1={v['f1-score']:.4f}")

per_class[["precision", "recall", "f1-score"]].plot.bar(figsize=(8, 4))
plt.title("Per-class precision / recall / F1 (held-out test set)")
plt.ylabel("score")
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

## 5. Confusion matrix

In [4]:
cm = confusion_matrix(yte, pred, labels=labels)
cm_df = pd.DataFrame(cm, index=[f"true {l}" for l in labels],
                     columns=[f"pred {l}" for l in labels])
print(cm_df.to_string())

plt.figure(figsize=(6, 5))
sns.heatmap(cm_df, annot=True, fmt="d", cmap="Blues")
plt.title("Confusion matrix — diet RandomForest (test set)")
plt.tight_layout()
plt.show()

NameError: name 'confusion_matrix' is not defined

## 6. Target distribution vs predicted distribution

In [ ]:
dist = pd.DataFrame({
    "train_target": ytr.value_counts(),
    "test_true": yte.value_counts(),
    "test_predicted": pd.Series(pred).value_counts(),
})
print(dist.to_string())
dist.plot.bar(figsize=(8, 4))
plt.title("Class balance: train vs test-true vs test-predicted")
plt.ylabel("count")
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()


## 7. Feature importance (from the trained forest, aggregated to base columns)

In [ ]:
# Importance straight from the forest trained above, aggregated
# from one-hot columns back to base features.
names = model.named_steps["preprocess"].get_feature_names_out()
imps = model.named_steps["clf"].feature_importances_
agg = {}
for _n, _v in zip(names, imps):
    _b = str(_n).split("__", 1)[-1]
    for _c in DIET_CATEGORICAL:
        if _b == _c or _b.startswith(_c + "_"):
            _b = _c
            break
    agg[_b] = agg.get(_b, 0.0) + float(_v)
_tot = sum(agg.values())
imp = pd.Series({k: v / _tot for k, v in agg.items()}).sort_values(ascending=False)
print("sums to:", round(float(imp.sum()), 6))
print(imp.round(4).to_string())
imp.sort_values().plot.barh(figsize=(8, 6))
plt.title("Feature importance — forest trained in this notebook")
plt.xlabel("importance")
plt.tight_layout()
plt.show()


## 8. Model vs dummy baseline (saved metrics)

In [ ]:
# Fresh dummy baseline trained on the SAME split in section 2.
from sklearn.metrics import precision_recall_fscore_support as _prf
dpred = dummy.predict(Xte)
rows = {}
for _name, _yp in [("random_forest", pred), ("dummy_stratified", dpred)]:
    _p, _r, _f, _ = _prf(yte, _yp, average="macro", zero_division=0)
    _pw, _rw, _fw, _ = _prf(yte, _yp, average="weighted", zero_division=0)
    rows[_name] = {"accuracy": accuracy_score(yte, _yp), "precision_macro": _p,
                   "recall_macro": _r, "f1_macro": _f, "f1_weighted": _fw}
comp = pd.DataFrame(rows)
print(comp.round(4).to_string())
comp.plot.bar(figsize=(8, 4))
plt.title("Freshly trained forest vs stratified dummy (same split)")
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()
print("\nTakeaway: the forest barely beats dummy — weak signal, informational use only.")


## 9. KNN food-catalog nutrition overview

In [ ]:
cat = pd.read_csv(REPO / "artifacts" / "indian_catalog.csv")
cols = ["energy_kcal", "proteins_100g", "fat_100g", "carbs_100g",
        "sugars_100g", "fiber_100g", "sodium_100g"]
print("rows:", len(cat))
print("missing %:", (cat[cols].isna().mean() * 100).round(1).to_dict())
cat[cols].hist(figsize=(12, 6), bins=30)
plt.suptitle("Indian catalog nutrition distributions (per 100 g)")
plt.tight_layout()
plt.show()